In [ ]:
# Data Audit and Dataset Integrity
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 200)
D = "../data/"
issues = []
def log(issue, rows, decision, reason):
    issues.append(dict(issue=issue, rows=int(rows), decision=decision, reason=reason))

In [ ]:
app  = pd.read_csv(D+"applications.csv", low_memory=False)
bur  = pd.read_csv(D+"credit_bureau.csv", low_memory=False)
srv  = pd.read_csv(D+"loan_servicing.csv", low_memory=False)
hold = pd.read_csv(D+"holdout_applications.csv", low_memory=False)
for n, d in [("applications",app),("bureau",bur),("servicing",srv),("holdout",hold)]:
    print(f"{n:13}", d.shape, "| unique ids:", d["id"].nunique(), "| exact dup rows:", d.duplicated().sum())

In [ ]:
# Inspect the repeated ids
dup = app[app.duplicated("id", keep=False)].sort_values("id")
print("repeated ids:", dup.id.nunique(), "| rows involved:", len(dup), "| extra rows:", len(app) - app.id.nunique())
print(dup.groupby("id").size().value_counts())
diff = dup.groupby("id").nunique(dropna=False).gt(1).sum()
print(diff[diff > 0].sort_values(ascending=False))
dup.head(6)

In [ ]:
# normalize employe titile  and drop repated id DTi measure the largest numberical d/f b/n records
a = dup.copy(); a["t"] = a.emp_title.str.strip().str.lower()
g = a.groupby("id")
print("emp_title differs beyond case:", (g.t.nunique(dropna=False) > 1).sum())
print("max dti gap:", round(g.dti.agg(lambda s: s.max() - s.min()).max(), 4))

log("same loan entered twice (emp_title case, dti +0.01)", 600, "drop 2nd row, keep first", "defaulted/amount/date identical; differences cosmetic")
log("emp_title inconsistent capitalization", 528, "fix", "normalize case/spaces; handled in A5 job grouping")
app = app.drop_duplicates("id", keep="first").reset_index(drop=True)
print(app.shape, app.id.is_unique)

In [ ]:
print("apps missing from bureau:", len(set(app.id) - set(bur.id)), "| from servicing:", len(set(app.id) - set(srv.id)))
df = (app.merge(bur, on="id", how="left", validate="one_to_one")
         .merge(srv, on="id", how="left", validate="one_to_one"))
print(df.shape)

In [ ]:
# map of the dataset
pd.DataFrame({"dtype": df.dtypes.astype(str),
              "missing_%": (df.isna().mean()*100).round(1),
              "n_unique": df.nunique()})

In [ ]:
# Validate date formats, loan terms,geographical values and identify inconsistent or non-standard categorical data.
def pattern(s): return s.dropna().str.replace(r"[A-Za-z]", "A", regex=True).str.replace(r"\d", "9", regex=True)
for c in ["issue_d", "earliest_cr_line", "last_pymnt_d"]:
    print(c); print(pattern(df[c]).value_counts()); print(df[c].sample(5, random_state=1).tolist(), "\n")
print(df.term.value_counts(dropna=False))
bad = df.addr_state[~df.addr_state.str.fullmatch(r"[A-Z]{2}")]
print("non-standard states:", len(bad)); print(bad.value_counts().head(15))
print("states after trim+upper:", df.addr_state.str.strip().str.upper().nunique())

In [ ]:
print(df.bureau_recovery_flag.value_counts())
print(df.groupby("bureau_recovery_flag").defaulted.agg(["mean", "count"]))
print(hold["bureau_recovery_flag"].value_counts())

In [ ]:
# parse dates and term
def parse_date(s):
    out = pd.to_datetime(s, format="%b-%Y", errors="coerce")
    for f in ["%Y-%m-%d", "%m/%Y"]:
        out = out.fillna(pd.to_datetime(s, format=f, errors="coerce"))
    return out
for c in ["issue_d", "earliest_cr_line"]:
    raw = df[c]; odd = (~raw.str.fullmatch(r"[A-Za-z]{3}-\d{4}")).sum()
    df[c] = parse_date(raw); fail = raw.notna().sum() - df[c].notna().sum()
    log(f"{c} mixed date formats (ISO, MM/YYYY)", odd, "fix", "parsed with explicit formats; 0 failures expected")
    print(c, "| odd formats:", odd, "| failed:", fail)
df["term"] = df.term.str.extract(r"(\d+)")[0].astype(int)
print(df.issue_d.min(), df.issue_d.max(), df.issue_d.nunique()); print(df.earliest_cr_line.min(), df.earliest_cr_line.max()); print(df.term.value_counts())

In [ ]:
# list the broken state values
s = df.addr_state.str.strip().str.upper()
bad = s[~s.str.fullmatch(r"[A-Z]{2}")]
print(bad.value_counts().to_string()); print("rows affected:", (df.addr_state != s).sum() + len(bad), "| valid codes:", s[s.str.fullmatch(r"[A-Z]{2}")].nunique())

In [ ]:
# log the leak and measure its AUC
from sklearn.metrics import roc_auc_score
SERV = [c for c in srv.columns if c != "id"]
LEAK = ["bureau_recovery_flag"] + SERV
log("bureau_recovery_flag: 100% default when 1, always 0 in holdout", 4207, "exclude", "recorded after outcome (label leak); AUC alone ~0.87")
log("all loan_servicing columns", len(df), "exclude from model, audit only", "recorded after issue (Mar 2019 snapshot)")
print("AUC of flag alone:", round(roc_auc_score(df.defaulted, df.bureau_recovery_flag), 3), "|", "bureau_recovery_flag" in hold.columns)

In [ ]:
# standardize the states
names = {"CALIFORNIA":"CA","TEXAS":"TX","NEW YORK":"NY","FLORIDA":"FL","GEORGIA":"GA","PENNSYLVANIA":"PA",
         "NEW JERSEY":"NJ","OHIO":"OH","ILLINOIS":"IL","NORTH CAROLINA":"NC","MICHIGAN":"MI","VIRGINIA":"VA"}
clean = df.addr_state.str.strip().str.upper()
s = clean.replace(names)
log("addr_state case/space/full-name inconsistencies", (s != df.addr_state).sum(), "fix", "trim, uppercase, map full names to codes")
print("rows changed:", (s != df.addr_state).sum(), "| full names:", clean.isin(names).sum(), "| states:", s.nunique(), "| all 2 letters:", s.str.fullmatch(r"[A-Z]{2}").all())
df["addr_state"] = s

In [ ]:
# impossible and suspicious value investigate 
rules = {
 "annual_inc <= 0": df.annual_inc <= 0,
 "0 < annual_inc < 1000 (thousands?)": (df.annual_inc > 0) & (df.annual_inc < 1000),
 "loan_amnt outside 1,000-40,000": (df.loan_amnt < 1000) | (df.loan_amnt > 40000),
 "int_rate outside 5-31": (df.int_rate < 5) | (df.int_rate > 31),
 "installment <= 0": df.installment <= 0,
 "dti < 0": df.dti < 0,
 "dti > 100": df.dti > 100,
 "revol_util > 100 (may be genuine)": df.revol_util > 100,
 "fico_range_low > fico_range_high": df.fico_range_low > df.fico_range_high,
 "earliest_cr_line after issue_d": df.earliest_cr_line > df.issue_d,
 "open_acc > total_acc": df.open_acc > df.total_acc,
 "pub_rec_bankruptcies > pub_rec": df.pub_rec_bankruptcies > df.pub_rec,
}
res = pd.Series({k: int(v.sum()) for k, v in rules.items()}, name="rows").to_frame()
res["%"] = (res.rows / len(df) * 100).round(2)
print(df.loc[rules["earliest_cr_line after issue_d"], "earliest_cr_line"].dt.year.value_counts().head())
res

In [ ]:
bad_inc = (df.annual_inc > 0) & (df.annual_inc < 1000)
print((df.loc[bad_inc, "annual_inc"] * 1000).describe().round(0)); print(df.loc[~bad_inc, "annual_inc"].describe().round(0))

r = df.int_rate / 1200; n = df.term
inst = df.loan_amnt * r / (1 - (1 + r) ** -n)
gap = (inst - df.installment).abs()
oor = (df.loan_amnt < 1000) | (df.loan_amnt > 40000)
print("installment mismatch >$1 | all:", round((gap > 1).mean(), 4), "| out-of-range loans:", round((gap[oor] > 1).mean(), 4))
print("below 1000:", (df.loan_amnt < 1000).sum(), "| above 40000:", (df.loan_amnt > 40000).sum()); print(df.loc[oor, "loan_amnt"].describe().round(0))

sw = df.fico_range_low > df.fico_range_high
print((df.fico_range_low - df.fico_range_high)[sw].value_counts().head())
print(((df.earliest_cr_line - df.issue_d).dt.days / 365)[df.earliest_cr_line > df.issue_d].describe().round(2))

In [ ]:
m = (df.annual_inc > 0) & (df.annual_inc < 1000)
log("annual_inc < 1000 (entered in thousands)", m.sum(), "fix: x1000", "distribution after x1000 matches the rest (median 63.5k vs 65k)")
df.loc[m, "annual_inc"] *= 1000

implied = df.installment * (1 - (1 + df.int_rate / 1200) ** -df.term) / (df.int_rate / 1200)
ratio = df.loan_amnt / implied
print("oor ratio:"); print(ratio[oor].describe().round(2))
print("recovered in range:", implied[oor].between(1000, 40000).mean().round(3), "| multiple of 25:", ((implied[oor] % 25).round(0).isin([0, 25])).mean().round(3))
mm = (gap > 1) & ~oor
print("other mismatches:", mm.sum()); print(df.loc[mm, ["loan_amnt", "installment", "int_rate", "term"]].assign(implied=implied[mm].round(0)).head(8))
print((df.fico_range_low - df.fico_range_high)[sw].value_counts().head(3)); print((df.fico_range_high - df.fico_range_low)[~sw].value_counts().head(3))

In [ ]:
r = df.int_rate / 1200
implied = df.installment * (1 - (1 + r) ** -df.term) / r
x10 = (df.loan_amnt / implied - 10).abs() < 0.05
print("x10 rows:", x10.sum(), "| hidden inside 1k-40k:", (x10 & (df.loan_amnt <= 40000)).sum())
log("loan_amnt inflated x10 (inconsistent with installment)", x10.sum(), "fix: divide by 10", "amount recomputed from installment/rate/term matches exactly; lands on multiples of 25")
df.loc[x10, "loan_amnt"] /= 10

sw = df.fico_range_low > df.fico_range_high
log("fico_range_low > fico_range_high (swapped)", sw.sum(), "fix: swap back", "gap is exactly 4 as in all normal rows")
df.loc[sw, ["fico_range_low", "fico_range_high"]] = df.loc[sw, ["fico_range_high", "fico_range_low"]].values

bad = df.earliest_cr_line > df.issue_d
log("earliest_cr_line after issue_d", bad.sum(), "set missing + flag cr_line_invalid", "impossible at application time; true date not recoverable")
df["cr_line_invalid"] = bad.astype(int); df.loc[bad, "earliest_cr_line"] = pd.NaT
print(df.loan_amnt.min(), df.loan_amnt.max(), "| swapped left:", (df.fico_range_low > df.fico_range_high).sum(), "| invalid credit lines:", bad.sum())

In [ ]:
r = df.int_rate / 1200; inst = df.loan_amnt * r / (1 - (1 + r) ** -df.term)
mm = (inst - df.installment).abs() > 1
print("remaining mismatches:", mm.sum()); print(df.loc[mm, "int_rate"].value_counts().head(8))
print(df.int_rate.value_counts().sort_index().head(8))

In [ ]:
from scipy.optimize import brentq
def solve_rate(P, pmt, n):
    f = lambda r: P*r/(1-(1+r)**-n) - pmt
    try: return brentq(f, 1e-6, 0.05) * 1200
    except ValueError: return np.nan
rows = df[mm].copy()
rows["solved_rate"] = [solve_rate(p, i, n) for p, i, n in zip(rows.loan_amnt, rows.installment, rows.term)]
rows["zero_rate_pmt"] = rows.loan_amnt / rows.term
typ = df[~mm].groupby("sub_grade").int_rate.agg(["min", "median", "max"])
print(rows[["loan_amnt", "installment", "zero_rate_pmt", "term", "int_rate", "solved_rate", "sub_grade"]].join(typ, on="sub_grade").round(2))

In [ ]:
for c in ["last_pymnt_d", "last_credit_pull_d"]:
    df[c] = pd.to_datetime(df[c], format="%b-%Y", errors="coerce")
checks = {
 "defaulted=0 but recoveries > 0": (df.defaulted == 0) & (df.recoveries > 0),
 "defaulted=0 but principal repaid < 99% of loan": (df.defaulted == 0) & (df.total_rec_prncp < 0.99 * df.loan_amnt),
 "defaulted=0 but debt_settlement_flag = Y": (df.defaulted == 0) & (df.debt_settlement_flag.astype(str).str.upper() == "Y"),
 "defaulted=1 but principal fully repaid": (df.defaulted == 1) & (df.total_rec_prncp >= 0.99 * df.loan_amnt),
 "last_pymnt_d before issue_d": df.last_pymnt_d < df.issue_d,
 "total_rec_prncp > loan_amnt": df.total_rec_prncp > df.loan_amnt * 1.01,
}
c = pd.Series({k: int(v.sum()) for k, v in checks.items()}, name="rows").to_frame(); c["%"] = (c.rows / len(df) * 100).round(2); c

In [ ]:
log("installment inconsistent with amount/rate/term; rate or installment contradicts sub_grade (8 at int_rate 6.00, 3 impossible installments)", mm.sum(), "drop", "no field provably correct, solved rate outside sub_grade range; 0.04% of rows")
log("label vs servicing: defaulted=1 but principal fully repaid", 1, "flag, keep", "plausible (unpaid interest/fees); other 5 cross-file checks found 0 contradictions")
df = df[~mm].reset_index(drop=True); print(len(df))

for c in ["grade", "sub_grade", "purpose", "home_ownership", "verification_status", "application_type", "debt_settlement_flag"]:
    s = df[c].astype(str); print(f"{c:22} raw: {s.nunique():3} | normalized: {s.str.strip().str.lower().nunique():3}")
print(df.home_ownership.value_counts()); print(df.purpose.value_counts())

issue_log = pd.DataFrame(issues); issue_log.to_csv("../reports/issue_log.csv", index=False)
df.to_parquet("../data/interim_clean.parquet", index=False); issue_log

In [ ]:
import pandas as pd
d = pd.read_parquet("../data/interim_clean.parquet"); il = pd.read_csv("../reports/issue_log.csv")
new = pd.DataFrame([
 {"issue": "revol_util negative (sign flipped)", "rows": int((d.revol_util < 0).sum()), "decision": "fix: absolute value", "reason": "abs value matches balance/limit recomputed (median gap 0.03, same as clean rows)"},
 {"issue": "bc_util negative (sign flipped)", "rows": int((d.bc_util < 0).sum()), "decision": "fix: absolute value", "reason": "same pattern; no limit column to verify, so an assumption"},
 {"issue": "dti above 100", "rows": int((d.dti > 100).sum()), "decision": "flag; spec valid range 0-100", "reason": "pre-2015 max is 39.97; probably an error, unproven"},
 {"issue": "fico_range_high - low != 4", "rows": int(((d.fico_range_high - d.fico_range_low) != 4).sum()), "decision": "flag, keep", "reason": "all other rows have a gap of exactly 4"}])
if not il.issue.isin(new.issue).any(): pd.concat([il, new]).to_csv("../reports/issue_log.csv", index=False)
print(pd.read_csv("../reports/issue_log.csv").tail(6))